In [1]:
# CELL 1 — INSTALL ALL DEPENDENCIES
# Run this FIRST. After it finishes → Runtime → Restart Runtime → continue from Cell 2.

!pip install ultralytics kagglehub roboflow gradio pandas pillow scikit-learn pyserial
print('✅ All packages installed! Now restart runtime.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 67.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 

In [2]:
# CELL 2 — CONNECT GOOGLE DRIVE
# This saves your model permanently. NEVER skip this.
# Click Allow when the permission popup appears.

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/GhostTrack'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Google Drive connected. Saving to: {SAVE_DIR}')

Mounted at /content/drive
✅ Google Drive connected. Saving to: /content/drive/MyDrive/GhostTrack


In [3]:
# CELL 6 — LOAD SAVED MODEL (use this next session instead of retraining)
# Run Cell 1 + Cell 2 first, then THIS cell.
# No retraining! Loads your saved model from Google Drive in seconds.

from ultralytics import YOLO
import shutil, os

MODEL_DRIVE = '/content/drive/MyDrive/GhostTrack/ghosttrack_best.pt'
MODEL_LOCAL = '/content/ghosttrack_best.pt'
YAML_DRIVE  = '/content/drive/MyDrive/GhostTrack/data.yaml'

if os.path.exists(MODEL_DRIVE):
    shutil.copy(MODEL_DRIVE, MODEL_LOCAL)
    if os.path.exists(YAML_DRIVE):
        shutil.copy(YAML_DRIVE, '/content/data.yaml')
    BEST_PT = MODEL_LOCAL
    print('✅ Model loaded from Google Drive!')
    print(f'   Path: {BEST_PT}')
    model = YOLO(BEST_PT)
    print('\nReady! You can now run Cell 7 (evaluate), Cell 9 (inference), or Cell 12 (demo).')
else:
    print('⚠️  No saved model found in Google Drive.')
    print('You need to run Cell 4 + Cell 5 first to train the model.')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Model loaded from Google Drive!
   Path: /content/ghosttrack_best.pt

Ready! You can now run Cell 7 (evaluate), Cell 9 (inference), or Cell 12 (demo).


In [4]:
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image

# Model 1 — COCO pretrained (handles person, bicycle, motorcycle perfectly)
model_coco = YOLO('yolov8n.pt')

# Model 2 — Your trained model (handles auto_rickshaw, tractor, Indian vehicles)
model_ghost = YOLO('/content/drive/MyDrive/GhostTrack/ghosttrack_best.pt')

# COCO class IDs we care about
COCO_CLASSES = {
    0:  'person',
    1:  'bicycle',
    3:  'motorcycle',
    2:  'car',
    5:  'bus',
    7:  'truck',
}

# Your model classes
GHOST_CLASSES = {
    0: 'person',
    1: 'bicycle',
    2: 'motorcycle',
    3: 'auto_rickshaw',
    4: 'e_rickshaw',
    5: 'car',
    6: 'lcv',
    7: 'bus',
    8: 'truck',
    9: 'tractor',
}

VRU_CLASSES = {'person', 'bicycle', 'motorcycle', 'auto_rickshaw', 'e_rickshaw'}

def ghosttrack_dual_detect(image_path):
    img = cv2.imread(image_path)

    all_detections = []

    # Run COCO model — gets person, bicycle, motorcycle
    results_coco = model_coco(img, conf=0.3, verbose=False)[0]
    for box in results_coco.boxes:
        cid = int(box.cls[0])
        if cid in COCO_CLASSES:
            all_detections.append({
                'class': COCO_CLASSES[cid],
                'conf':  float(box.conf[0]),
                'box':   box.xyxy[0].tolist(),
                'source': 'COCO'
            })

    # Run your model — gets auto_rickshaw, tractor, Indian vehicles
    results_ghost = model_ghost(img, conf=0.3, verbose=False)[0]
    for box in results_ghost.boxes:
        cid = int(box.cls[0])
        cname = GHOST_CLASSES.get(cid, 'unknown')
        # Only add Indian-specific classes (avoid duplicate person/car from your model)
        if cname in ['auto_rickshaw', 'e_rickshaw', 'tractor', 'lcv']:
            all_detections.append({
                'class': cname,
                'conf':  float(box.conf[0]),
                'box':   box.xyxy[0].tolist(),
                'source': 'GHOST'
            })

    # Print results
    print(f"\nTotal detections: {len(all_detections)}")
    for d in all_detections:
        vru = "⚠️ VRU" if d['class'] in VRU_CLASSES else "  "
        print(f"  {vru} {d['class']} | conf: {d['conf']:.2f} | source: {d['source']}")

    return all_detections

# Test it
ghosttrack_dual_detect('/content/test_road.jpg')

WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.
WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.

Total detections: 4
     bus | conf: 0.87 | source: COCO
  ⚠️ VRU person | conf: 0.87 | source: COCO
  ⚠️ VRU person | conf: 0.85 | source: COCO
  ⚠️ VRU person | conf: 0.83 | source: COCO


[{'class': 'bus',
  'conf': 0.8734484910964966,
  'box': [22.87126922607422,
   231.2772674560547,
   805.002685546875,
   756.8403930664062],
  'source': 'COCO'},
 {'class': 'person',
  'conf': 0.8656908869743347,
  'box': [48.55046844482422,
   398.5522155761719,
   245.34556579589844,
   902.7026977539062],
  'source': 'COCO'},
 {'class': 'person',
  'conf': 0.852835476398468,
  'box': [669.472900390625,
   392.1860656738281,
   809.7201538085938,
   877.0354614257812],
  'source': 'COCO'},
 {'class': 'person',
  'conf': 0.8252246975898743,
  'box': [221.517333984375,
   405.79864501953125,
   344.9706726074219,
   857.53662109375],
  'source': 'COCO'}]

In [5]:
import yaml

yaml_config = {
    'path':  '/content/ghosttrack_dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc':    10,
    'names': {
        0: 'person',
        1: 'bicycle',
        2: 'motorcycle',
        3: 'auto_rickshaw',
        4: 'e_rickshaw',
        5: 'car',
        6: 'lcv',
        7: 'bus',
        8: 'truck',
        9: 'tractor'
    }
}

with open('/content/data.yaml', 'w') as f:
    yaml.dump(yaml_config, f, sort_keys=False)

print('✅ data.yaml created!')

✅ data.yaml created!


In [ ]:
# CELL 7 — EXPORT TO ONNX FOR RASPBERRY PI 4
# Converts model to ONNX format for Pi 4 deployment.

from ultralytics import YOLO
import shutil

model     = YOLO(BEST_PT)
model.export(format='onnx', imgsz=640, simplify=True)

ONNX_PATH = BEST_PT.replace('.pt', '.onnx')
shutil.copy(ONNX_PATH, f'{SAVE_DIR}/ghosttrack_best.onnx')
print(f'✅ ONNX saved to: {SAVE_DIR}/ghosttrack_best.onnx')
print('Copy ghosttrack_best.onnx to Raspberry Pi 4 when components arrive.')

In [ ]:
# CELL 8 — GHOSTTRACK DUAL MODEL INFERENCE ENGINE
# Uses 2 models combined:
#   Model 1 — YOLOv8n COCO  → person, bicycle, motorcycle, car, bus, truck
#   Model 2 — Your model    → auto_rickshaw, e_rickshaw, tractor, lcv

import csv, datetime, numpy as np
from ultralytics import YOLO

GHOST_LOG_PATH = '/content/ghost_log.csv'
ESP32_SERIAL   = None   # Change to '/dev/ttyUSB0' on Pi 4

# ── Load both models ──────────────────────────────────────────────────
model_coco  = YOLO('yolov8n.pt')           # COCO pretrained
model_ghost = YOLO(BEST_PT)                # Your GhostTrack model

# ── Class definitions ─────────────────────────────────────────────────
COCO_CLASSES = {
    0: 'person',
    1: 'bicycle',
    3: 'motorcycle',
    2: 'car',
    5: 'bus',
    7: 'truck'
}

GHOST_CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'motorcycle',
    3: 'auto_rickshaw', 4: 'e_rickshaw',
    5: 'car', 6: 'lcv', 7: 'bus', 8: 'truck', 9: 'tractor'
}

# Only these classes trigger haptic alert
VRU_CLASSES = {'person', 'bicycle', 'motorcycle', 'auto_rickshaw', 'e_rickshaw'}

# ── Distance estimator ────────────────────────────────────────────────
def estimate_distance(box_h_ratio):
    if box_h_ratio > 0.40:   return 'CRITICAL', 1.5
    elif box_h_ratio > 0.20: return 'NEAR',     3.5
    else:                    return 'FAR',       7.0

# ── Danger side ───────────────────────────────────────────────────────
def get_danger_side(x_ratio):
    return 'LEFT' if x_ratio < 0.5 else 'RIGHT'

# ── ESP32 signal ──────────────────────────────────────────────────────
def send_esp32_signal(side, zone):
    signal = f'ALERT:{side}:{zone}'
    if ESP32_SERIAL:
        import serial
        with serial.Serial(ESP32_SERIAL, 115200, timeout=1) as s:
            s.write((signal + '\n').encode())
    else:
        print(f'  📡 [SIM] ESP32 → {signal}')

# ── Ghost Log ─────────────────────────────────────────────────────────
ghost_ready = False
def write_ghost_log(ts, cls, dist, zone, side, conf):
    global ghost_ready
    if not ghost_ready:
        with open(GHOST_LOG_PATH, 'w', newline='') as f:
            csv.writer(f).writerow(
                ['timestamp','class','distance_m','zone','side','confidence','alert_type']
            )
        ghost_ready = True
    with open(GHOST_LOG_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([
            ts, cls, f'{dist:.1f}', zone, side,
            f'{conf:.2f}', 'CRITICAL' if zone=='CRITICAL' else 'AWARENESS'
        ])

# ── Main dual inference function ──────────────────────────────────────
def ghosttrack_inference(frame):
    all_dets = []
    h, w = frame.shape[:2]

    # Run COCO model — person, bicycle, motorcycle, car, bus, truck
    r1 = model_coco(frame, conf=0.3, verbose=False)[0]
    for box in r1.boxes:
        cid = int(box.cls[0])
        if cid in COCO_CLASSES:
            all_dets.append({
                'class':  COCO_CLASSES[cid],
                'conf':   float(box.conf[0]),
                'box':    box.xyxy[0].tolist(),
                'source': 'COCO'
            })

    # Run Ghost model — auto_rickshaw, e_rickshaw, tractor, lcv only
    r2 = model_ghost(frame, conf=0.3, verbose=False)[0]
    for box in r2.boxes:
        cid   = int(box.cls[0])
        cname = GHOST_CLASSES.get(cid, 'unknown')
        # Only add Indian-specific classes to avoid duplicates
        if cname in ['auto_rickshaw', 'e_rickshaw', 'tractor', 'lcv']:
            all_dets.append({
                'class':  cname,
                'conf':   float(box.conf[0]),
                'box':    box.xyxy[0].tolist(),
                'source': 'GHOST'
            })

    alerts = []
    ts     = datetime.datetime.now().isoformat()

    for d in all_dets:
        x1, y1, x2, y2 = d['box']
        bh    = (y2 - y1) / h
        xc    = ((x1 + x2) / 2) / w
        zone, dist = estimate_distance(bh)
        side       = get_danger_side(xc)
        cname      = d['class']

        # Log everything
        write_ghost_log(ts, cname, dist, zone, side, d['conf'])

        # Alert only for VRUs
        if cname in VRU_CLASSES:
            alerts.append({
                'class': cname, 'side': side,
                'distance_m': dist, 'zone': zone,
                'confidence': d['conf'], 'source': d['source']
            })
            if zone in ('CRITICAL', 'NEAR'):
                send_esp32_signal(side, zone)
            print(
                f"  ⚠️  {cname.upper()} | {side} | "
                f"{dist:.1f}m ({zone}) | conf:{d['conf']:.2f} | [{d['source']}]"
            )

    # Use r1 annotated frame for display
    return r1.plot(), alerts

print('✅ Dual model inference engine ready!')
print('   COCO model  → person, bicycle, motorcycle, car, bus, truck')
print('   Ghost model → auto_rickshaw, e_rickshaw, tractor, lcv')
print('\nRun Cell 10 to test on an image.')

In [ ]:
# CELL 9A — TEST ON VIDEO (with annotated video output)

import cv2
import os

VIDEO_PATH   = '/content/videoplayback.mp4'
OUTPUT_PATH  = '/content/ghosttrack_output.mp4'
FRAME_SKIP   = 1      # process every frame (set higher to skip, but output will have gaps)

if not os.path.exists(VIDEO_PATH):
    print('⚠️  Video not found! Check the path.')
else:
    cap          = cv2.VideoCapture(VIDEO_PATH)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'Video   : {VIDEO_PATH}')
    print(f'Frames  : {total_frames}  |  FPS: {fps:.1f}  |  Size: {width}x{height}')
    print(f'Output  : {OUTPUT_PATH}\n')

    # ── Video writer ──────────────────────────────────────────────────────────
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

    frame_idx      = 0
    all_alerts_log = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_SKIP == 0:
            # Run GhostTrack inference on this frame
            annotated, alerts = ghosttrack_inference(frame)

            if alerts:
                for a in alerts:
                    print(f"  ⚠️  Frame {frame_idx} | {a['class']} | {a['side']} | {a['distance_m']}m | {a['zone']}")
                all_alerts_log.extend(alerts)

            out.write(annotated)   # write annotated frame
        else:
            out.write(frame)       # write original frame (keeps video timing intact)

        frame_idx += 1

        # Progress every 100 frames
        if frame_idx % 100 == 0:
            pct = (frame_idx / total_frames) * 100
            print(f'  Progress: {frame_idx}/{total_frames} frames ({pct:.1f}%)')

    cap.release()
    out.release()

    print(f'\n══ Summary ══════════════════════════════')
    print(f'Frames processed : {frame_idx}')
    print(f'Total VRU alerts : {len(all_alerts_log)}')
    print(f'Output saved     : {OUTPUT_PATH}')

    # ── Re-encode for Colab playback (mp4v → H.264) ──────────────────────────
    FINAL_PATH = '/content/ghosttrack_final.mp4'
    os.system(f'ffmpeg -y -i {OUTPUT_PATH} -vcodec libx264 -crf 23 {FINAL_PATH}')
    print(f'Playable video   : {FINAL_PATH}')

In [ ]:
# CELL 10A — Play output video in Colab

from IPython.display import HTML
from base64 import b64encode
import os

FINAL_PATH = '/content/ghosttrack_final.mp4'

# Check if file exists and has content
if os.path.exists(FINAL_PATH):
    size_mb = os.path.getsize(FINAL_PATH) / (1024 * 1024)
    print(f'File size: {size_mb:.2f} MB')

    with open(FINAL_PATH, 'rb') as f:
        video_data = f.read()

    video_b64 = b64encode(video_data).decode()

    display(HTML(f'''
        <video width="800" controls autoplay>
            <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        </video>
    '''))
else:
    print('❌ Video not found. Check if ffmpeg re-encode finished successfully.')
    # Try the raw output instead
    RAW_PATH = '/content/ghosttrack_output.mp4'
    if os.path.exists(RAW_PATH):
        print(f'Raw output exists: {os.path.getsize(RAW_PATH)/(1024*1024):.2f} MB — trying that instead...')
        os.system(f'ffmpeg -y -i {RAW_PATH} -vcodec libx264 -crf 23 {FINAL_PATH}')
        print('Re-encode done. Re-run this cell.')

In [ ]:
# ── CELL 9B — LIVE LAPTOP WEBCAM WITH DUAL YOLO TRACKING & LOGS ──────────

import cv2
import numpy as np
import datetime
import pandas as pd
import os
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from ultralytics import YOLO
import PIL.Image
import io

# 1. LOAD BOTH MODELS
model_coco  = YOLO('yolov8n.pt')
model_ghost = YOLO(BEST_PT)

COCO_CLASSES = {0: 'person', 1: 'bicycle', 3: 'motorcycle', 2: 'car', 5: 'bus', 7: 'truck'}
GHOST_CLASSES = {0:'person', 1:'bicycle', 2:'motorcycle', 3:'auto_rickshaw', 4:'e_rickshaw',
                 5:'car', 6:'lcv', 7:'bus', 8:'truck', 9:'tractor'}
VRU_CLASSES = {'person', 'bicycle', 'motorcycle', 'auto_rickshaw', 'e_rickshaw'}

# Direct styling configurations matching your palette
COLORS = {
    'person': (136, 45, 255, 255),       # Pinkish-Red
    'bicycle': (0, 149, 255, 255),      # Orange
    'motorcycle': (48, 59, 255, 255),   # Red
    'auto_rickshaw': (214, 86, 88, 255), # Purple
    'e_rickshaw': (222, 82, 175, 255),   # Light Violet
    'car': (89, 199, 52, 255),          # Green
    'bus': (190, 199, 0, 255),          # Teal
    'truck': (30, 28, 28, 255),         # Dark Grey
    'lcv': (102, 99, 99, 255),          # Grey
    'tractor': (94, 132, 162, 255)      # Brown
}

# 2. JAVASCRIPT WEBCAM STREAM INJECTOR MECHANISM
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var overlayCanvas;
    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
      stream.getVideoTracks()[0].stop();
      video.remove();
      div.remove();
      video = null; div = null; stream = null; captureCanvas = null; overlayCanvas = null;
    }

    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid #00ff88';
      div.style.padding = '4px';
      div.style.position = 'relative';
      div.style.width = '640px';
      document.body.appendChild(div);

      const titleLabel = document.createElement('div');
      titleLabel.innerHTML = "<b>👻 GHOSTTRACK LIVE WORKSPACE (REAL-TIME LOGGING)</b>";
      titleLabel.style.color = "#00ff88";
      titleLabel.style.marginBottom = "5px";
      div.appendChild(titleLabel);

      const container = document.createElement('div');
      container.style.position = 'relative';
      container.style.width = '640px';
      container.style.height = '480px';
      div.appendChild(container);

      video = document.createElement('video');
      video.style.position = 'absolute';
      video.style.zIndex = '1';
      video.width = 640;
      video.height = 480;
      video.autoplay = true;
      video.playsInline = true;
      container.appendChild(video);

      overlayCanvas = document.createElement('canvas');
      overlayCanvas.style.position = 'absolute';
      overlayCanvas.style.zIndex = '2';
      overlayCanvas.width = 640;
      overlayCanvas.height = 480;
      container.appendChild(overlayCanvas);

      stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: "user"}});
      video.srcObject = stream;

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;

      window.requestAnimationFrame(onAnimationFrame);
      return stream;
    }

    async function streamFrame() {
      if (div === null) { await createDom(); }
      return new Promise((resolve) => { pendingResolve = resolve; });
    }

    function drawBBox(imgData) {
      if (shutdown || !overlayCanvas) return;
      var ctx = overlayCanvas.getContext('2d');
      ctx.clearRect(0, 0, 640, 480);
      if (imgData === "") return;

      var img = new Image();
      img.onload = function() {
        ctx.drawImage(img, 0, 0);
      };
      img.src = imgData;
    }

    async function stopStream() {
      shutdown = true;
      removeDom();
    }

    window.streamFrame = streamFrame;
    window.drawBBox = drawBBox;
    window.stopStream = stopStream;
  ''')
  display(js)

def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

def bbox_to_bytes(bbox_array):
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  io_buf = io.BytesIO()
  bbox_PIL.save(io_buf, format='png')
  bbox_bytes = 'data:image/png;base64,{}'.format((b64encode(io_buf.getvalue()).decode('utf-8')))
  return bbox_bytes

import PIL.Image
import io

# 3. LIVE RUNTIME INFERENCE LOOP WITH YOUR ORIGINAL LOGGER FORMAT
video_stream()
print(f"[OK] Streaming active. Telemetry outputs logging below...\n")

frame_idx = 0
live_session_alerts = []

try:
  while True:
    js_reply = eval_js('streamFrame()')
    if not js_reply:
      break

    frame = js_to_image(js_reply)
    all_dets = []

    # ── MODEL 1: COCO ──
    r1 = model_coco.track(source=frame, persist=True, verbose=False)[0]
    if r1.boxes is not None:
        for box in r1.boxes:
            cid = int(box.cls[0])
            if cid in COCO_CLASSES:
                all_dets.append({
                    'class': COCO_CLASSES[cid],
                    'conf': float(box.conf[0]),
                    'box': box.xyxy[0].tolist(),
                    'source_model': 'COCO'
                })

    # ── MODEL 2: GHOST ──
    r2 = model_ghost.track(source=frame, persist=True, verbose=False)[0]
    if r2.boxes is not None:
        for box in r2.boxes:
            cid = int(box.cls[0])
            cname = GHOST_CLASSES.get(cid, 'unknown')
            if cname in ['auto_rickshaw', 'e_rickshaw', 'tractor', 'lcv']:
                all_dets.append({
                    'class': cname,
                    'conf': float(box.conf[0]),
                    'box': box.xyxy[0].tolist(),
                    'source_model': 'GHOST'
                })

    bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)
    frame_vrus_logged = False

    # Process and build log data blocks for this active frame
    for d in all_dets:
        x1, y1, x2, y2 = d['box']
        col = COLORS.get(d['class'], (255, 255, 255, 255))

        # Geometry processing rules
        xc = ((x1 + x2) / 2) / 640.0
        bh = (y2 - y1) / 480.0

        zone = 'CRITICAL' if bh > 0.4 else 'NEAR' if bh > 0.2 else 'FAR'
        side = 'LEFT' if xc < 0.5 else 'RIGHT'
        dist = 1.5 if zone == 'CRITICAL' else 3.5 if zone == 'NEAR' else 7.0

        # Render the graphics overlay strings onto the browser mask
        cv2.rectangle(bbox_array, (int(x1), int(y1)), (int(x2), int(y2)), col, 3)
        label = f"{d['class']} {d['conf']:.2f}"
        cv2.putText(bbox_array, label, (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)

        # Is this target classified as a VRU?
        if d['class'] in VRU_CLASSES:
            # 🚨 1. PRINT YOUR ORIGINAL SIMULATED ESP32 COMMS MESSAGE STRING 🚨
            if not frame_vrus_logged:
                print(f"📡  [SIM] ESP32 → ALERT:{side}:{zone}")
                frame_vrus_logged = True

            # 🚨 2. PRINT YOUR PRECISE DYNAMIC CLASSIFICATION DETAILS ROW 🚨
            print(f"⚠️   {d['class'].upper()} | {side} | {dist:.1f}m ({zone}) | conf:{d['conf']:.2f} | [{d['source_model']}]")

            # Save data row variables into your global notebook state
            log_entry = {
                'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'class': d['class'],
                'confidence': d['conf'],
                'side': side,
                'zone': zone,
                'distance_m': dist,
                'alert_type': zone
            }
            live_session_alerts.append(log_entry)

    # Push layout drawings onto browser view panel
    bbox_bytes = bbox_to_bytes(bbox_array)
    eval_js(f'drawBBox("{bbox_bytes}")')

    frame_idx += 1

except KeyboardInterrupt:
  print('\n[OK] Stopping live feed interpretation safely.')

  # Append findings to GHOST_LOG_PATH on exit
  if live_session_alerts:
      new_df = pd.DataFrame(live_session_alerts)
      if os.path.exists(GHOST_LOG_PATH):
          existing_df = pd.read_csv(GHOST_LOG_PATH)
          combined_df = pd.concat([existing_df, new_df], ignore_index=True)
          combined_df.to_csv(GHOST_LOG_PATH, index=False)
      else:
          new_df.to_csv(GHOST_LOG_PATH, index=False)
      print(f'\n══ Session Finished ══════════════════════')
      print(f'Ghost Log updated successfully at: {GHOST_LOG_PATH}')

finally:
  eval_js('stopStream()')

In [ ]:
# ── REWRITTEN CELL: VIEW RAW BOX COORDINATES FROM YOUR LIVE WEBCAM ──

import os

print('All detections from your last active live frame:')
print('──────────────────────────────────────────────────')

# Check if we have active detections stored from your webcam session
if 'all_dets' in globals() and len(all_dets) > 0:
    names = ['person','bicycle','motorcycle','auto_rickshaw',
             'e_rickshaw','car','lcv','bus','truck','tractor']

    for d in all_dets:
        cname = d['class']
        conf = d['conf']
        x1, y1, x2, y2 = d['box']

        # Print out the exact text layouts matching your project criteria
        print(f'  {cname} | conf: {conf:.2f} | box: {int(x1)},{int(y1)},{int(x2)},{int(y2)} | [{d["source_model"]}]')
else:
    print('⚠️ No active live detections found in memory!')
    print('👉 Make sure to run Cell 10, let the camera detect your face, and hit the Stop ⏹️ button before running this cell.')

In [ ]:
# CELL 11 — VIEW GHOST LOG
# Shows all detection events saved during testing.
# This is the same data that saves to MicroSD on the real truck.

import pandas as pd, os

VRU_CLASSES = {'person', 'bicycle', 'motorcycle', 'auto_rickshaw', 'e_rickshaw'}

if os.path.exists(GHOST_LOG_PATH):
    df = pd.read_csv(GHOST_LOG_PATH)
    print(f'Ghost Log — {len(df)} total events\n')
    print(df.tail(20).to_string(index=False))
    print('\n── Detections by class ──────────────────')
    print(df['class'].value_counts().to_string())
    print('\n── VRU alerts by side ───────────────────')
    vru_df = df[df['class'].isin(VRU_CLASSES)]
    print(vru_df['side'].value_counts().to_string() if len(vru_df) > 0 else 'No VRU alerts yet.')
    print('\n── By distance zone ─────────────────────')
    print(df['zone'].value_counts().to_string())
    print('\n── Critical alerts only ─────────────────')
    crit = df[df['alert_type'] == 'CRITICAL']
    print(f'Total critical: {len(crit)}')
    if len(crit) > 0:
        print(crit[['timestamp','class','distance_m','side']].to_string(index=False))
else:
    print('⚠️  No Ghost Log found. Run Cell 10 first.')

In [ ]:
# CELL 12(only for A) — GRADIO DEMO WITH DUAL MODEL
import gradio as gr
from ultralytics import YOLO
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# Load both models
model_coco  = YOLO('yolov8n.pt')
model_ghost = YOLO(BEST_PT)

COCO_CLASSES = {
    0: 'person', 1: 'bicycle', 3: 'motorcycle',
    2: 'car', 5: 'bus', 7: 'truck'
}
GHOST_CLASSES = {
    0:'person', 1:'bicycle', 2:'motorcycle',
    3:'auto_rickshaw', 4:'e_rickshaw',
    5:'car', 6:'lcv', 7:'bus', 8:'truck', 9:'tractor'
}
VRU_CLASSES = {'person','bicycle','motorcycle','auto_rickshaw','e_rickshaw'}

# Colors per class
COLORS = {
    'person':'#FF2D55', 'bicycle':'#FF9500',
    'motorcycle':'#FF3B30', 'auto_rickshaw':'#5856D6',
    'e_rickshaw':'#AF52DE', 'car':'#34C759',
    'bus':'#00C7BE', 'truck':'#1C1C1E',
    'tractor':'#A2845E', 'lcv':'#636366'
}

def dual_detect(image):
    img_array = np.array(image)
    all_dets  = []

    # COCO model — person, bicycle, motorcycle, car, bus, truck
    r1 = model_coco(img_array, conf=0.3, verbose=False)[0]
    for box in r1.boxes:
        cid = int(box.cls[0])
        if cid in COCO_CLASSES:
            all_dets.append({
                'class': COCO_CLASSES[cid],
                'conf':  float(box.conf[0]),
                'box':   box.xyxy[0].tolist()
            })

    # Ghost model — auto_rickshaw, e_rickshaw, tractor, lcv
    r2 = model_ghost(img_array, conf=0.3, verbose=False)[0]
    for box in r2.boxes:
        cid   = int(box.cls[0])
        cname = GHOST_CLASSES.get(cid, 'unknown')
        if cname in ['auto_rickshaw','e_rickshaw','tractor','lcv']:
            all_dets.append({
                'class': cname,
                'conf':  float(box.conf[0]),
                'box':   box.xyxy[0].tolist()
            })

    # Draw boxes on image
    draw = ImageDraw.Draw(image)
    alerts = []
    vru_found = False

    for d in all_dets:
        x1,y1,x2,y2 = d['box']
        col  = COLORS.get(d['class'], '#FFFFFF')
        draw.rectangle([x1,y1,x2,y2], outline=col, width=3)
        draw.text((x1+4, y1+4), f"{d['class']} {d['conf']:.2f}", fill=col)

        # Estimate distance and side
        bh   = (y2-y1)/img_array.shape[0]
        xc   = ((x1+x2)/2)/img_array.shape[1]
        zone = 'CRITICAL' if bh>0.4 else 'NEAR' if bh>0.2 else 'FAR'
        side = 'LEFT' if xc < 0.5 else 'RIGHT'
        dist = 1.5 if zone=='CRITICAL' else 3.5 if zone=='NEAR' else 7.0

        if d['class'] in VRU_CLASSES:
            vru_found = True
            e = '🔴' if zone=='CRITICAL' else '🟡'
            alerts.append(
                f"{e} {d['class'].upper()}\n"
                f"   Side: {side} blind spot\n"
                f"   Distance: ~{dist:.1f}m ({zone})\n"
                f"   Confidence: {d['conf']:.0%}\n"
                f"   ESP32 → ALERT:{side}:{zone}"
            )

    if not vru_found:
        status = '✅ SAFE — No VRUs in blind spot'
    elif any('CRITICAL' in a for a in alerts):
        status = '🔴 CRITICAL ALERT — Haptic warning firing!'
    else:
        status = '🟡 AWARENESS — VRU detected, monitoring...'

    output = status
    if alerts:
        output += '\n\n' + '\n\n'.join(alerts)

    return image, output

gr.Interface(
    fn=dual_detect,
    inputs=gr.Image(type='pil', label='Upload blind spot image'),
    outputs=[
        gr.Image(type='pil', label='Detection Result'),
        gr.Textbox(label='GhostTrack Alert Status', lines=16),
    ],
    title='🚛 GhostTrack — Dual Model Detection System',
    description='YOLOv8n COCO (person/bicycle/motorcycle) + GhostTrack model (auto_rickshaw/tractor) fused together.',
).launch(share=True)